In [1]:
!pip install -U torch torchvision torchaudio
!pip install -U transformers accelerate peft bitsandbytes trl datasets qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.5/915.5 MB 69.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 100.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 73.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 96.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 95.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 106.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 112.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.

In [7]:
import torch
import gc

# 1. Zero out references (more effective than just 'del' in some cases)
model = None
processor = None

# 2. Garbage collect first to ensure Python-side references are gone
gc.collect()

# 3. Synchronize before emptying to ensure all kernels have finished
torch.cuda.synchronize()

# 4. Empty the cache
torch.cuda.empty_cache()


## Setup and Model Assembly

In [2]:
import os
import re
import json
import torch
from PIL import Image
from tqdm import tqdm
import json_repair
from peft import PeftModel
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info

# --- Configuration ---
BASE_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
ADAPTER_PATH = "/workspace/qwen_lora_final/qwen_lora_final" # Update if path changed
IMG_DIR = "local_images"

# Route HF downloads to workspace
os.environ["HF_HOME"] = "/workspace/huggingface_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"

# --- Load Model & Processor ---
print(f"Loading Base Processor from {BASE_MODEL_ID}...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

print("Loading Base Model into VRAM...")
base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

print(f"Attaching LoRA Adapter from {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print("Merging weights for ultra-fast inference...")
model = model.merge_and_unload()
model.eval()
print("✅ Model successfully assembled in memory!\n")

Loading Base Processor from Qwen/Qwen2.5-VL-7B-Instruct...


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`torch_dtype` is deprecated! Use `dtype` instead!


Loading Base Model into VRAM...


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Attaching LoRA Adapter from /workspace/qwen_lora_final/qwen_lora_final...
Merging weights for ultra-fast inference...
✅ Model successfully assembled in memory!



## Inference Extraction Logic

In [3]:
def run_qwen_inference(image_path):
    """
    Runs an image through the model, cleans common syntax hallucinations, 
    and extracts the JSON object.
    """
    schema_prompt = "Extract the Extended ChartSpec JSON from this chart image."
    
    messages = [
        {"role": "system", "content": "You output strictly valid JSON with no markdown formatting or extra text."},
        {"role": "user", "content": [
            {"type": "image", "image": image_path, "max_pixels": 768 * 768},
            {"type": "text", "text": schema_prompt}
        ]}
    ]

    # Process Inputs
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate Output (Clean generation)
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=4096,
            temperature=0.1,  
            do_sample=False
        )

    # Decode Text
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    prediction_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    
    # --- REGEX CLEANING ---
    # Fixes specific hallucination where the model over-generates closing brackets before the relation array
    clean_json_str = re.sub(r'\}\]\}\],\"rel\"', '}],\"rel\"', prediction_text)
    
    # Safely parse JSON
    try:
        raw_spec = json_repair.repair_json(clean_json_str, return_objects=True)
    except Exception:
        raw_spec = {}

    return raw_spec

## Constraint Enforcement

In [4]:
def process_extracted_spec(raw_spec, metadata):
    """
    Injects the title and strictly enforces schema constraints 
    before feeding to Stage 2.
    """
    if not isinstance(raw_spec, dict):
        return {}
        
    # 1. Title Injection
    # raw_spec['title'] = metadata.get('title_description', '')
    chart_type = metadata.get('chart_type', '').lower()
    
    # 2. Strict Schema Enforcement 
    for series in raw_spec.get('ser', []):
        # Pie charts must NOT produce 'trend' and 'stats'
        if 'pie_chart' in chart_type:
            series.pop('trend', None)
            series.pop('stats', None)
            
        # Box plots must use 'stats' instead of 'distribution_summary'
        if 'box' in chart_type:
            if 'distribution_summary' in series:
                series['stats'] = series.pop('distribution_summary')
                
    return raw_spec

## Main Loop & Dataset Generation

In [6]:
import os
import json
from PIL import Image
from tqdm import tqdm

input_datasets = [
    "train_wo_spec.json", 
    "validation_wo_spec.json", 
    "test_1_wo_spec.json", 
    "test_2_wo_spec.json"
]

CACHE_FILE = "global_spec_cache.json"

def safe_convert_to_jpg(img_path):
    """
    Safely converts PNGs to JPGs. If the PNG has a transparent background,
    it pastes the chart onto a white background first so the text doesn't turn black.
    """
    jpg_path = img_path.rsplit(".", 1)[0] + ".jpg"
    
    if not os.path.exists(jpg_path):
        try:
            img = Image.open(img_path)
            # Check for transparency
            if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
                # Standardize to RGBA to ensure we always have exactly 4 channels
                img = img.convert('RGBA')
                # Create a solid white background
                background = Image.new('RGB', img.size, (255, 255, 255))
                # Passing 'img' as the mask tells PIL to automatically use the Alpha channel
                background.paste(img, mask=img) 
                background.save(jpg_path, 'JPEG', quality=95)
            else:
                img.convert('RGB').save(jpg_path, 'JPEG', quality=95)
        except Exception as e:
            print(f"Failed to convert {img_path} to JPG: {e}")
            return img_path # Fallback to original if conversion fails
            
    return jpg_path

# --- NEW: Load or Initialize the Global Spec Cache ---
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'r', encoding='utf-8') as f:
        global_spec_cache = json.load(f)
    print(f"🧠 Loaded Global Cache with {len(global_spec_cache)} unique images.")
else:
    global_spec_cache = {}
    print("🧠 Initialized new empty Global Cache.")


for dataset_file in input_datasets:
    if not os.path.exists(dataset_file):
        print(f"Skipping {dataset_file} (Not found)")
        continue
        
    print(f"\n🚀 Processing Dataset: {dataset_file}")
    
    with open(dataset_file, 'r', encoding='utf-8') as f:
        data_rows = json.load(f)
        
    output_filename = dataset_file.replace("_wo_spec.json", "_w_spec.json")
    compiled_data = []
    
    # Dataset-level Resume Logic
    if os.path.exists(output_filename):
        try:
            with open(output_filename, 'r', encoding='utf-8') as f:
                compiled_data = json.load(f)
            print(f"🔄 Resuming: Found {len(compiled_data)} existing rows in {output_filename}.")
        except json.JSONDecodeError:
            print(f"⚠️ Warning: {output_filename} is corrupted. Starting from scratch.")
            compiled_data = []

    start_index = len(compiled_data)
    remaining_rows = data_rows[start_index:]
    
    if not remaining_rows:
        print(f"✅ Dataset {dataset_file} is already 100% complete!")
        continue

    missing_images_count = 0
    
    for row in tqdm(remaining_rows, desc=f"Inferencing {dataset_file}"):
        raw_img_path = row.get("local_image_path", "")
        
        # OS Pathing Fix (Windows to Linux)
        filename = raw_img_path.replace("\\", "/").split("/")[-1]
        img_path = os.path.join("local_images", filename)
        
        # Safe JPG Conversion
        if img_path.lower().endswith(".png") and os.path.exists(img_path):
            img_path = safe_convert_to_jpg(img_path)
            
        row["local_image_path"] = img_path 
            
        if os.path.exists(img_path):
            try:
                # --- NEW: Check Cache Before Running Inference ---
                if img_path in global_spec_cache:
                    # Instant retrieval!
                    raw_spec = global_spec_cache[img_path]
                else:
                    # Run Model & Update Cache
                    raw_spec = run_qwen_inference(img_path)
                    global_spec_cache[img_path] = raw_spec
                    
                    # Save cache to disk to protect progress
                    with open(CACHE_FILE, 'w', encoding='utf-8') as f:
                        json.dump(global_spec_cache, f)
                
                # 2. Clean & Enforce Rules
                clean_spec = process_extracted_spec(raw_spec, row)
                
                # 3. Inject description
                if isinstance(clean_spec, dict):
                    clean_spec["description"] = row.get("title_description", "")
                
                # 4. Attach to Row
                row["extended_spec"] = clean_spec
                
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                row["extended_spec"] = {}
        else:
            if missing_images_count < 5:
                print(f"\n⚠️ WARNING: Image not found at path -> '{img_path}'")
            missing_images_count += 1
            row["extended_spec"] = {}
            
        compiled_data.append(row)
        
        # Save dataset row progress
        with open(output_filename, 'w', encoding='utf-8') as f:
            json.dump(compiled_data, f, indent=4)
        
    if missing_images_count > 0:
        print(f"❌ Skipped {missing_images_count} rows because images were missing.")
        
    print(f"✅ Finished! Saved {len(compiled_data)} total rows to {output_filename}")

print("\n🎉 All datasets successfully compiled for Stage 2 Training!")

🧠 Loaded Global Cache with 268 unique images.

🚀 Processing Dataset: train_wo_spec.json
🔄 Resuming: Found 334 existing rows in train_w_spec.json.


Inferencing train_wo_spec.json:  11%|█         | 784/7273 [2:02:44<4:27:03,  2.47s/it] 

Failed to convert local_images/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.png to JPG: cannot identify image file 'local_images/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.png'
Error processing local_images/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.png: cannot identify image file 'local_images/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.png'


Inferencing train_wo_spec.json:  17%|█▋        | 1237/7273 [2:54:19<39:35:57, 23.62s/it]

Failed to convert local_images/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.png to JPG: cannot identify image file 'local_images/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.png'
Error processing local_images/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.png: cannot identify image file 'local_images/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.png'


...

Error processing local_images/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.jpg: cannot identify image file 'local_images/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.jpg'


Inferencing test_1_wo_spec.json: 100%|██████████| 939/939 [00:44<00:00, 21.23it/s]


✅ Finished! Saved 939 total rows to test_1_w_spec.json

🚀 Processing Dataset: test_2_wo_spec.json


Inferencing test_2_wo_spec.json: 100%|██████████| 981/981 [40:41<00:00,  2.49s/it]  

✅ Finished! Saved 981 total rows to test_2_w_spec.json

🎉 All datasets successfully compiled for Stage 2 Training!


## Isolate and Fix the Bad Images

In [7]:
import json

dataset_files = ["train_w_spec.json", "validation_w_spec.json", "test_1_w_spec.json", "test_2_w_spec.json"]

for file in dataset_files:
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    missing = [row for row in data if not row.get("extended_spec")]
    
    if missing:
        print(f"\n--- Missing in {file}: {len(missing)} rows ---")
        for row in missing[:5]: # Print first 5 to keep it clean
            print(f"File: {row.get('local_image_path')}")
            # Assuming your dataset has the original url stored under 'image_url' or similar
            print(f"URL: {row.get('original_url', 'URL not found in row')}\n")


--- Missing in train_w_spec.json: 4 rows ---
File: local_images/RespondentsByGroup.jpg
URL: https://chartfc.s3.amazonaws.com/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.png

File: local_images/RespondentsByGroup.jpg
URL: https://chartfc.s3.amazonaws.com/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.png

File: local_images/OverallUsefulness.jpg
URL: https://chartfc.s3.amazonaws.com/Media_Viewer_-_Survey_Graph_-_Overall_Usefulness_-_May_5_2014.png

File: local_images/OverallUsefulness.jpg
URL: https://chartfc.s3.amazonaws.com/Media_Viewer_-_Survey_Graph_-_Overall_Usefulness_-_May_5_2014.png


--- Missing in test_1_w_spec.json: 2 rows ---
File: local_images/OverallUsefulness.jpg
URL: https://chartfc.s3.amazonaws.com/Media_Viewer_-_Survey_Graph_-_Overall_Usefulness_-_May_5_2014.png

File: local_images/RespondentsByGroup.jpg
URL: https://chartfc.s3.amazonaws.com/Media_Viewer_-_Survey_Graph_-_Respondents_by_Group_-_May_5_2014.png



## The Refill Script

In [8]:
# --- THE REFILL PIPELINE ---
# Run this AFTER fixing the corrupted images in your folder.

for dataset_file in dataset_files:
    with open(dataset_file, 'r', encoding='utf-8') as f:
        data_rows = json.load(f)
        
    fixed_count = 0
    
    for row in tqdm(data_rows, desc=f"Refilling {dataset_file}"):
        # Only process rows that currently have an empty spec
        if not row.get("extended_spec"):
            img_path = row.get("local_image_path", "")
            
            # Safe JPG Conversion Phase (in case the new file is a transparent PNG)
            if img_path.lower().endswith(".png") and os.path.exists(img_path):
                img_path = safe_convert_to_jpg(img_path)
                row["local_image_path"] = img_path 
                
            if os.path.exists(img_path):
                try:
                    # Check cache or run inference
                    if img_path in global_spec_cache:
                        raw_spec = global_spec_cache[img_path]
                    else:
                        raw_spec = run_qwen_inference(img_path)
                        global_spec_cache[img_path] = raw_spec
                        
                        with open(CACHE_FILE, 'w', encoding='utf-8') as cache_f:
                            json.dump(global_spec_cache, cache_f)
                            
                    clean_spec = process_extracted_spec(raw_spec, row)
                    if isinstance(clean_spec, dict):
                        clean_spec["description"] = row.get("title_description", "")
                        
                    row["extended_spec"] = clean_spec
                    fixed_count += 1
                    
                except Exception as e:
                    print(f"Still failing on {img_path}: {e}")
                    
    # Save the patched dataset back to disk
    if fixed_count > 0:
        with open(dataset_file, 'w', encoding='utf-8') as f:
            json.dump(data_rows, f, indent=4)
        print(f"✅ Successfully refilled {fixed_count} missing specs in {dataset_file}!")
    else:
        print(f"⚡ No specs needed refilling in {dataset_file}.")

Refilling train_w_spec.json: 100%|██████████| 7607/7607 [00:00<00:00, 3169372.26it/s]


✅ Successfully refilled 4 missing specs in train_w_spec.json!


Refilling validation_w_spec.json: 100%|██████████| 953/953 [00:00<00:00, 2464347.54it/s]


⚡ No specs needed refilling in validation_w_spec.json.


Refilling test_1_w_spec.json: 100%|██████████| 939/939 [00:00<00:00, 1379492.63it/s]

✅ Successfully refilled 2 missing specs in test_1_w_spec.json!



Refilling test_2_w_spec.json: 100%|██████████| 981/981 [00:00<00:00, 3606145.68it/s]

⚡ No specs needed refilling in test_2_w_spec.json.


In [2]:
import json

def prune_spec_for_encoder(raw_spec_dict):
    """
    Minifies the JSON string, intelligently downsamples massive data arrays, 
    and uses a recursive global sweeper to kill VLM autoregressive loops ANYWHERE in the JSON.
    """
    if not isinstance(raw_spec_dict, dict):
        return "{}"
        
    cleaned_spec = raw_spec_dict.copy()
    
    # --- [FIX 4] THE GHOST SPEC GATEKEEPER ---
    # If the chart has no series, it is mathematically dead. 
    # Reject it immediately so the description field doesn't mask it.
    if 'ser' not in cleaned_spec or not isinstance(cleaned_spec['ser'], list) or len(cleaned_spec['ser']) == 0:
        return "{}"
        
    # --- HELPER 1: Safe Float Compression ---
    def compress_val(v):
        """Rounds floats to 2 decimals, converts .0 floats to ints, leaves strings alone."""
        if isinstance(v, float):
            return int(v) if v.is_integer() else round(v, 2)
        return v

    # --- HELPER 2: The Universal Loop Killer ---
    def kill_repeating_loops(lst, max_consecutive=2, max_length=20):
        """Removes consecutive duplicates and hard-caps length at 20."""
        if not isinstance(lst, list): return lst
        cleaned_list = []
        last_item = None
        consecutive_count = 0
        
        for item in lst:
            if item == last_item:
                consecutive_count += 1
            else:
                last_item = item
                consecutive_count = 1
                
            if consecutive_count <= max_consecutive:
                cleaned_list.append(compress_val(item))
                
        return cleaned_list[:max_length]

    # --- HELPER 3: Global List Sweeper (Updated for Lists of Dicts) ---
    def sweep_for_loops(obj):
        """Recursively hunts down any list in the JSON and applies the Loop Killer."""
        if isinstance(obj, dict):
            for k, v in list(obj.items()):
                if k == 'data': 
                    continue # Leave raw data arrays to the explicit downsampler below
                if isinstance(v, list):
                    # If it's a list of primitives (strings/numbers), kill loops
                    if all(isinstance(x, (str, int, float, bool)) for x in v):
                        obj[k] = kill_repeating_loops(v)
                    # If it's a list of dicts, recurse deeper into them
                    else:
                        for item in v:
                            sweep_for_loops(item)
                elif isinstance(v, dict):
                    sweep_for_loops(v)

    # 1. Ensure structural conformity & nuke bloated relation arrays
    cleaned_spec.pop('distribution_summary', None)
    cleaned_spec.pop('legend', None)
    cleaned_spec.pop('rel', None) # [FIX 1] Destroy hallucinated/bloated rankings
    
    topo_dict = cleaned_spec.get('topo', {})
    raw_topo_type = topo_dict.get('type', '') if isinstance(topo_dict, dict) else ''
    topo_type = raw_topo_type.lower() if isinstance(raw_topo_type, str) else ''
    
    # 2. Globally sweep and kill loops in ALL fields
    sweep_for_loops(cleaned_spec)
    
    # --- [FIX 3] MASSIVE CHART TRIGGER ---
    series_list = cleaned_spec.get('ser', [])
    is_massive_chart = len(series_list) > 3 
    
    # 3. Explicitly Sub-sample Series Data safely 
    for ser in series_list:
        if isinstance(ser, dict):
            
            # [FIX 2] Remove extraneous fields that slipped through
            ser.pop('ds', None)
            ser.pop('is_subsampled', None)
            ser.pop('critical_points_retained', None)
            ser.pop('y_ref', None) 
            
            # If it's a pie chart OR has too many lines, strip redundant stats/trends
            if 'pie' in topo_type or is_massive_chart:
                ser.pop('trend', None)
                ser.pop('stats', None)
            
            if 'data' in ser and isinstance(ser['data'], list):
                raw_data = ser['data']
                
                # Downsample to max 20 data points
                if len(raw_data) > 20:
                    indices = [int(i * (len(raw_data) - 1) / 19) for i in range(20)]
                    raw_data = [raw_data[i] for i in indices]
                    
                # Compress the float values inside the data array
                compressed_data = []
                for point in raw_data:
                    if isinstance(point, list):
                        compressed_data.append([compress_val(v) for v in point])
                    else:
                        compressed_data.append(compress_val(point))
                ser['data'] = compressed_data

    # Return minified JSON string
    return json.dumps(cleaned_spec, separators=(',', ':'))

In [11]:
import os
import json
import torch
from tqdm import tqdm
from transformers import AutoTokenizer
from qwen_vl_utils import process_vision_info
import json_repair

# ==========================================
# --- 1. Global Configuration ---
# ==========================================
MODEL_NAME = "microsoft/deberta-v3-base" # Upgraded to DeBERTa for Disentangled Attention
MAX_CLAIM_LEN = 128
MAX_SPEC_LEN = 1024  # Bumped up from 512
BATCH_SIZE = 8       # Dropped from 16 to fit the 1024 context window in VRAM
EPOCHS = 4
LEARNING_RATE = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Re-inference Threshold mapped to your DeBERTa limit
TOKEN_THRESHOLD = MAX_SPEC_LEN 

# ==========================================
# --- 2. Setup & Cache Loading ---
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset_files = ["train_w_spec.json", "validation_w_spec.json", "test_1_w_spec.json", "test_2_w_spec.json"]
CACHE_FILE = "global_spec_cache.json"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'r', encoding='utf-8') as f:
        global_spec_cache = json.load(f)
else:
    global_spec_cache = {}
    print("⚠️ No global_spec_cache.json found. Creating a new one.")

# ==========================================
# --- 3. Anti-Loop Inference Function ---
# ==========================================
def run_qwen_anti_loop(image_path):
    """
    Re-runs inference with repetition penalties and higher temperature 
    to break autoregressive VLM loops.
    """
    schema_prompt = (
        "Extract the Extended ChartSpec JSON from this chart image. "
        "CRITICAL: Do not repeat the same categories endlessly. Limit data arrays to essential points."
    )
    
    messages = [
        {"role": "system", "content": "You output strictly valid, concise JSON with no markdown formatting. Never repeat the same array items in an infinite loop."},
        {"role": "user", "content": [
            {"type": "image", "image": image_path, "max_pixels": 768 * 768},
            {"type": "text", "text": schema_prompt}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=4096,    # Lower max tokens to prevent runaway generation
            temperature=0.4,        # Introduce slight randomness to break the loop
            do_sample=True,         # Required for temperature to work
            repetition_penalty=1.15 # Severely penalize repeating the same token
        )

    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    prediction_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    
    # Clean common bracket hallucination
    clean_json_str = prediction_text.replace('}]}],\"rel\"', '}],\"rel\"')
    clean_json_str = clean_json_str.replace('}],\"rel\"', '}],\"rel\"')
    
    try:
        raw_spec = json_repair.repair_json(clean_json_str, return_objects=True)
    except Exception:
        raw_spec = {}
        
    return raw_spec

# ==========================================
# --- 4. Surgical Execution Loop ---
# ==========================================
# Create a temporary cache to track what we fix in real-time
fixed_images_this_run = {}

for dataset_file in dataset_files:
    if not os.path.exists(dataset_file):
        continue
        
    with open(dataset_file, 'r', encoding='utf-8') as f:
        data_rows = json.load(f)
        
    fixed_count = 0
    print(f"\n🔍 Scanning {dataset_file} for outliers > {TOKEN_THRESHOLD} tokens...")
    
    for row in tqdm(data_rows, desc=f"Checking {dataset_file}"):
        
        raw_extended = row.get("extended_spec", {})
        
        # 1. Temporarily prune to check the TRUE length it will have in Stage 2
        test_pruned_str = prune_spec_for_encoder(raw_extended)
        tokens = tokenizer.encode(test_pruned_str)
        
        # 2. Smart Re-Inference Trigger
        needs_reinference = False
        
        if len(tokens) > TOKEN_THRESHOLD:
            needs_reinference = True
        elif not raw_extended or 'ser' not in raw_extended or not raw_extended['ser']:
            needs_reinference = True
            
        if needs_reinference:
            img_path = row.get("local_image_path", "")
            
            if os.path.exists(img_path):
                try:
                    # --- NEW: Check if we ALREADY fixed this image ---
                    if img_path in fixed_images_this_run:
                        new_raw_spec = fixed_images_this_run[img_path]
                    else:
                        # Only run VLM if it's a new unique outlier
                        new_raw_spec = run_qwen_anti_loop(img_path)
                        
                        # Overwrite the bad cache entry and mark as fixed
                        global_spec_cache[img_path] = new_raw_spec
                        fixed_images_this_run[img_path] = new_raw_spec
                    
                    # Clean and verify the new extraction
                    clean_spec = process_extracted_spec(new_raw_spec, row)
                    if isinstance(clean_spec, dict):
                        clean_spec["description"] = row.get("title_description", "")
                    
                    # Save the new clean dict back to the row
                    row["extended_spec"] = clean_spec
                    fixed_count += 1
                except Exception as e:
                    print(f"Failed surgical fix on {img_path}: {e}")

    # Save progress back to disk
    if fixed_count > 0:
        with open(CACHE_FILE, 'w', encoding='utf-8') as f:
            json.dump(global_spec_cache, f)
            
        with open(dataset_file, 'w', encoding='utf-8') as f:
            json.dump(data_rows, f, indent=4)
        print(f"✅ Surgically fixed {fixed_count} rows in {dataset_file}!")
    else:
        print(f"⚡ No bloated charts found above {TOKEN_THRESHOLD} tokens in {dataset_file}.")

print("\n🎉 Surgical Re-Inference Complete! You are ready for Stage 2.")


🔍 Scanning train_w_spec.json for outliers > 1024 tokens...


Checking train_w_spec.json: 100%|██████████| 7607/7607 [2:22:58<00:00,  1.13s/it]   


✅ Surgically fixed 2008 rows in train_w_spec.json!

🔍 Scanning validation_w_spec.json for outliers > 1024 tokens...


Checking validation_w_spec.json: 100%|██████████| 953/953 [00:00<00:00, 1995.88it/s]


✅ Surgically fixed 272 rows in validation_w_spec.json!

🔍 Scanning test_1_w_spec.json for outliers > 1024 tokens...


Checking test_1_w_spec.json: 100%|██████████| 939/939 [00:00<00:00, 2066.44it/s]


✅ Surgically fixed 252 rows in test_1_w_spec.json!

🔍 Scanning test_2_w_spec.json for outliers > 1024 tokens...


Checking test_2_w_spec.json: 100%|██████████| 981/981 [14:37<00:00,  1.12it/s]


✅ Surgically fixed 272 rows in test_2_w_spec.json!

🎉 Surgical Re-Inference Complete! You are ready for Stage 2.


In [6]:
# --- THE GHOST SPEC HUNTER (Run in spec_extraction.ipynb) ---
dataset_files = ["train_w_spec.json", "validation_w_spec.json", "test_1_w_spec.json", "test_2_w_spec.json"]
CACHE_FILE = "global_spec_cache.json"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'r', encoding='utf-8') as f:
        global_spec_cache = json.load(f)
else:
    global_spec_cache = {}
    print("⚠️ No global_spec_cache.json found. Creating a new one.")
for dataset_file in dataset_files:
    with open(dataset_file, 'r', encoding='utf-8') as f:
        data_rows = json.load(f)
        
    fixed_count = 0
    for row in tqdm(data_rows, desc=f"Hunting Ghosts in {dataset_file}"):
        
        raw_extended = row.get("extended_spec", {})
        
        # TRIGGER: If the spec is empty OR it is missing the data series
        if not raw_extended or 'ser' not in raw_extended or not raw_extended['ser']:
            img_path = row.get("local_image_path", "")
            
            if os.path.exists(img_path):
                try:
                    new_raw_spec = None
                    
                    # 1. Check if the global cache ALREADY has a healthy version of this spec
                    if img_path in global_spec_cache:
                        cached_spec = global_spec_cache[img_path]
                        if isinstance(cached_spec, dict) and 'ser' in cached_spec and cached_spec['ser']:
                            new_raw_spec = cached_spec
                            
                    # 2. If no healthy version exists, force a fresh extraction & overwrite bad cache
                    if new_raw_spec is None:
                        new_raw_spec = run_qwen_inference(img_path) 
                        global_spec_cache[img_path] = new_raw_spec
                    
                    # 3. Clean and inject description
                    clean_spec = process_extracted_spec(new_raw_spec, row)
                    if isinstance(clean_spec, dict):
                        clean_spec["description"] = row.get("title_description", "")
                        
                    row["extended_spec"] = clean_spec
                    fixed_count += 1
                except Exception as e:
                    print(f"Failed on {img_path}: {e}")

    # Save patched dataset and updated cache back to disk
    if fixed_count > 0:
        with open(CACHE_FILE, 'w', encoding='utf-8') as f:
            json.dump(global_spec_cache, f)
            
        with open(dataset_file, 'w', encoding='utf-8') as f:
            json.dump(data_rows, f, indent=4)
        print(f"✅ Resurrected {fixed_count} ghost specs in {dataset_file}!")
    else:
        print(f"⚡ No ghost specs found in {dataset_file}.")

Hunting Ghosts in train_w_spec.json: 100%|██████████| 7607/7607 [34:37<00:00,  3.66it/s]  


✅ Resurrected 106 ghost specs in train_w_spec.json!


Hunting Ghosts in validation_w_spec.json: 100%|██████████| 953/953 [04:35<00:00,  3.46it/s]


✅ Resurrected 19 ghost specs in validation_w_spec.json!


Hunting Ghosts in test_1_w_spec.json: 100%|██████████| 939/939 [03:24<00:00,  4.59it/s]


✅ Resurrected 17 ghost specs in test_1_w_spec.json!


Hunting Ghosts in test_2_w_spec.json: 100%|██████████| 981/981 [00:00<00:00, 3369870.78it/s]

⚡ No ghost specs found in test_2_w_spec.json.
